In [9]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv('../data/customer_churn.csv')
print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [11]:
# Fix TotalCharges column without chained assignment warning
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())

# Convert Target Variable (Churn) to binary
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# Drop customerID column
if 'customerID' in df.columns:
    df = df.drop(columns=['customerID'])

# Categorical variables to One-Hot Encoding
df = pd.get_dummies(df, drop_first=True)

print("Cleaned Data Shape:", df.shape)

Cleaned Data Shape: (7043, 31)


In [12]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['Churn'])
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("Training set shape:", X_train.shape)
print("Testing set shape:", X_test.shape)

Training set shape: (5634, 30)
Testing set shape: (1409, 30)


In [15]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score

# Train XGBoost model
model = XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=5, random_state=42)
model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

# Metrics
print("Classification Report:")
print(classification_report(y_test, y_pred))
print("ROC-AUC Score:", roc_auc_score(y_test, y_proba))

Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.90      0.87      1035
           1       0.66      0.54      0.59       374

    accuracy                           0.80      1409
   macro avg       0.75      0.72      0.73      1409
weighted avg       0.80      0.80      0.80      1409

ROC-AUC Score: 0.8451006225942287


In [14]:
!pip install xgboost

   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
    --------------------------------------- 1.0/48.9 MB 8.5 MB/s eta 0:00:06
   -- ------------------------------------- 2.6/48.9 MB 8.2 MB/s eta 0:00:06
   --- ------------------------------------ 4.5/48.9 MB 8.3 MB/s eta 0:00:06
   ----- ---------------------------------- 6.3/48.9 MB 8.4 MB/s eta 0:00:06
   ------ --------------------------------- 8.1/48.9 MB 8.5 MB/s eta 0:00:05
   -------- ------------------------------- 10.0/48.9 MB 8.5 MB/s eta 0:00:05
   --------- ------------------------------ 11.3/48.9 MB 8.3 MB/s eta 0:00:05
   ---------- ----------------------------- 12.8/48.9 MB 8.3 MB/s eta 0:00:05
   ----------- ---------------------------- 14.4/48.9 MB 8.3 MB/s eta 0:00:05
   ------------- -------------------------- 16.0/48.9 MB 8.1 MB/s eta 0:00:05
   -------------- ------------------------- 17.8/48.9 MB 8.1 MB/s eta 0:00:04
   ---------------- ----------------------- 19.7/48.9 MB 8.1 MB/s eta 0:00:04

In [16]:
import joblib
import os

os.makedirs('../models', exist_ok=True)
joblib.dump(model, '../models/xgboost_churn_model.pkl')
print("Model successfully saved to ../models/xgboost_churn_model.pkl")

Model successfully saved to ../models/xgboost_churn_model.pkl
